# Sesión 5 · Visualización: funciones que dibujan

**Programación avanzada en Python aplicada al análisis social** · miércoles 16 de diciembre de 2026

Objetivos:

1. Entender las dos piezas de un gráfico de Matplotlib: la figura y los ejes.
2. Escribir **funciones que dibujan**, para repetir un gráfico con otros datos sin copiar y pegar.
3. Convertir un análisis en una pequeña **aplicación** con Streamlit.

El diseño de visualizaciones se trabaja a fondo en la optativa *Visualización de datos sociales digitales*. Aquí nos centramos en la parte de programación: gráficos correctos, repetibles y fáciles de mantener.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
hipotecas.head(3)

## 1. El gráfico rápido de pandas

Para un primer vistazo basta el método `.plot()` de pandas.

In [ ]:
salamanca = hipotecas[hipotecas["provincia"] == "Salamanca"]
salamanca.plot(x="fecha", y="hipotecas");

Sirve para explorar, pero no para enseñárselo a nadie: no dice qué mide ni de dónde sale.

## 2. Figura y ejes

Matplotlib trabaja con dos objetos:

- la **figura** (`fig`): el lienzo completo, lo que se guarda en un fichero;
- los **ejes** (`ax`): cada gráfico dentro de la figura, con su título, sus etiquetas y sus líneas.

`plt.subplots()` crea ambos. Después, todo se pide a `ax`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

ax.plot(salamanca["fecha"], salamanca["hipotecas"])
ax.set_title("Hipotecas sobre viviendas en Salamanca")
ax.set_ylabel("Hipotecas al mes")
ax.set_ylim(0)                                       # el eje vertical empieza en cero

plt.show()

### Suavizar una serie

La serie mensual tiene mucho ruido. La **media móvil** de 12 meses (la media de cada mes con los once anteriores) deja ver la tendencia.

In [ ]:
serie = salamanca.set_index("fecha")["hipotecas"]
suavizada = serie.rolling(12).mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(serie.index, serie, color="lightgray", label="Dato mensual")
ax.plot(suavizada.index, suavizada, color="tab:blue", linewidth=2, label="Media móvil de 12 meses")
ax.set_title("Hipotecas sobre viviendas en Salamanca")
ax.set_ylabel("Hipotecas al mes")
ax.set_ylim(0)
ax.legend()
plt.show()

## 3. Una función que dibuja

¿Y el mismo gráfico para Granada? ¿Y para Soria? En lugar de copiar la celda, la convertimos en función. La provincia pasa a ser un **parámetro**.

In [ ]:
def grafico_provincia(hipotecas, provincia, ax=None):
    """Dibuja la serie mensual de hipotecas de una provincia y su media móvil de 12 meses."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 3.5))

    serie = hipotecas[hipotecas["provincia"] == provincia].set_index("fecha")["hipotecas"]
    ax.plot(serie.index, serie, color="lightgray", label="Dato mensual")
    ax.plot(serie.index, serie.rolling(12).mean(), color="tab:blue", linewidth=2, label="Media móvil de 12 meses")

    ax.set_title(f"Hipotecas sobre viviendas en {provincia}")
    ax.set_ylabel("Hipotecas al mes")
    ax.set_ylim(0)
    return ax


grafico_provincia(hipotecas, "Granada");

El parámetro `ax=None` es un truco útil: si no se indica, la función crea su propia figura; si se le pasa uno, **dibuja donde se le diga**. Eso permite componer varios gráficos:

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.2))

for ax, provincia in zip(ejes, ["Salamanca", "Granada", "Madrid"]):
    grafico_provincia(hipotecas, provincia, ax=ax)

ejes[0].legend(fontsize=8)
fig.tight_layout()

Tres gráficos idénticos en forma, con seis líneas de código. Y si mañana queréis cambiar un color, se cambia **en un solo sitio**.

### Guardar la figura

In [ ]:
Path("resultados").mkdir(exist_ok=True)
fig.savefig("resultados/hipotecas_tres_provincias.png", dpi=200, bbox_inches="tight")

> **Lista de comprobación de un gráfico**
>
>  - El título dice qué se mide, dónde y cuándo.
> - Los ejes llevan unidades.
> - En barras y áreas, el eje empieza en cero.
> - Al comparar territorios de tamaño distinto se usan tasas, no totales.
> - Se cita la fuente.

## 4. Comparar con seaborn

seaborn dibuja directamente a partir de una tabla en formato largo: se le dice qué columna va en cada eje y cuál define el color.

In [ ]:
import shutil

# El módulo 'vivienda.py' lo escribimos en la sesión 4. Si no está en la carpeta de trabajo, lo copiamos.
if not Path("vivienda.py").exists():
    shutil.copy("sesiones/s04_datos_tabulares_2/vivienda.py", "vivienda.py")

In [ ]:
import seaborn as sns

import vivienda

tabla = vivienda.anadir_tasa(vivienda.resumen_anual(vivienda.leer_hipotecas()))
dos = tabla[tabla["provincia"].isin(["Salamanca", "Granada"])]

ax = sns.lineplot(data=dos, x="anyo", y="tasa_mil", hue="provincia", marker="o")
ax.set(title="Hipotecas anuales por cada mil habitantes", xlabel="", ylabel="Por mil habitantes", ylim=(0, None));

Fijaos en la segunda línea: hemos reutilizado el módulo `vivienda` de la sesión anterior. Para eso sirve ordenar el código.

## 5. Un gráfico interactivo con Plotly

Con Plotly el gráfico responde al ratón: muestra los valores, permite ampliar y ocultar series.

In [ ]:
import plotly.express as px

varias = hipotecas[hipotecas["provincia"].isin(["Salamanca", "Granada", "Valladolid", "Málaga"])].copy()
varias["media_movil"] = varias.groupby("provincia")["hipotecas"].transform(lambda s: s.rolling(12).mean())

fig = px.line(varias, x="fecha", y="media_movil", color="provincia",
              title="Hipotecas sobre viviendas (media móvil de 12 meses)",
              labels={"media_movil": "Hipotecas al mes", "fecha": "", "provincia": "Provincia"})
fig.show()

## 6. De un notebook a una aplicación: Streamlit

Streamlit convierte un fichero de Python en una página web con controles. La idea es sencilla: **cada vez que alguien mueve un control, el fichero se ejecuta entero de arriba abajo** con el nuevo valor.

```python
import pandas as pd
import plotly.express as px
import streamlit as st

st.title("Hipotecas sobre viviendas por provincia")

hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])

provincias = sorted(hipotecas["provincia"].unique())
elegidas = st.sidebar.multiselect("Provincias", provincias, default=["Salamanca", "Granada"])

datos = hipotecas[hipotecas["provincia"].isin(elegidas)]
st.plotly_chart(px.line(datos, x="fecha", y="hipotecas", color="provincia"))
```

Son las mismas tres piezas de siempre: **leer**, **filtrar** y **dibujar**. Lo único nuevo es que el filtro lo decide quien usa la aplicación.

La plantilla completa está en `apps/app_plantilla.py`. Streamlit **no funciona en Colab**; hay que ejecutarlo en vuestro ordenador:

```bash
uv run streamlit run apps/app_plantilla.py
```

## Resumen

- `fig, ax = plt.subplots()`; después, todo se pide a `ax`.
- Si vais a repetir un gráfico, hacedlo **función**, con `ax=None` para poder componer.
- seaborn para tablas largas; Plotly cuando la interacción aporta.
- Una aplicación de Streamlit es un fichero que lee, filtra y dibuja.